# Fruta-database vullen in Snowflake

Laadt de Hugging Face-datasets en schrijft ze naar `FRUTA.PUBLIC`. Draait lokaal en maakt verbinding met de Snowflake-gegevens uit `.env`.

In [ ]:
import os

import pandas as pd
from datasets import Dataset, load_dataset
from dotenv import load_dotenv
from snowflake.snowpark import Session

# Zet op True om bestaande tabellen te overschrijven
OVERWRITE = False

# Hugging Face-dataset -> tabelnaam in Snowflake
HF_DATASETS = {
    "ekrombouts/fruta_profiles": "PROFILES",
    "ekrombouts/fruta_scenarios": "SCENARIOS",
    "ekrombouts/fruta_records": "RECORDS",
    # Afterthought: voorbeelden van rapportages, niet gelinkt aan clientprofielen,
    # maar op veel voorkomende onderwerpen.
    "ekrombouts/gardenia_notes": "NOTES",
}

In [ ]:
# Laadt een Hugging Face-dataset en converteert deze naar een Pandas DataFrame
def dataset_to_df(dataset_name, split="all") -> pd.DataFrame:
    ds = load_dataset(path=dataset_name, split=split)
    assert isinstance(ds, Dataset), "ds is not a Hugging Face Dataset"
    return ds.to_pandas()


# Controleert of een tabel al bestaat in het huidige schema
def table_exists(session, table_name) -> bool:
    return len(session.sql(f"SHOW TABLES LIKE '{table_name}'").collect()) > 0


# Schrijft een DataFrame naar Snowflake; kolomnamen worden hoofdletters (Snowflake-conventie)
def write_to_snowflake(session, df, table_name):
    df.columns = [col.upper() for col in df.columns]
    session.write_pandas(
        df,
        table_name,
        auto_create_table=True,
        overwrite=True,
        quote_identifiers=False,
    )
    print(f"Tabel '{table_name}' geschreven: {len(df)} rijen.")

In [ ]:
load_dotenv()
session = Session.builder.configs(
    {
        "account": os.getenv("SNOWFLAKE_ACCOUNT"),
        "user": os.getenv("SNOWFLAKE_USER"),
        "password": os.getenv("SNOWFLAKE_PASSWORD"),
        "warehouse": "COMPUTE_WH",
        "database": "FRUTA",
        "schema": "PUBLIC",
    }
).create()

In [ ]:
for dataset_name, table_name in HF_DATASETS.items():
    if table_exists(session, table_name) and not OVERWRITE:
        print(f"Overgeslagen: '{table_name}' bestaat al.")
        continue

    df = dataset_to_df(dataset_name)
    if table_name == "NOTES":
        df["id"] = df.index + 1
    write_to_snowflake(session, df, table_name)

In [ ]:
# Controle: aantal rijen per tabel
for table_name in HF_DATASETS.values():
    print(table_name, session.table(table_name).count())

In [ ]:
session.close()